# Portion Estimation Pipeline — Food-101
## Segmentation: SAM-b (Segment Anything Model Base)
> Pipeline: Preprocess → SAM-b Segment → Extract 5 Ratios → Weighted Score → S/M/L

In [ ]:
# Cài đặt SAM (chỉ cần chạy 1 lần)
!pip install ultralytics --quiet
!pip install supervision --quiet  # optional, dùng để visualise

In [ ]:
import os
import cv2
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay)
from ultralytics import SAM

# ---------- Đường dẫn Kaggle ----------
DATA_ROOT  = Path("/kaggle/input/datasets/minhquang2701/food101-15cl")
IMAGE_DIR  = DATA_ROOT
OUTPUT_DIR = Path("/kaggle/working/portion_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- 15 class ----------
CLASSES = [
    "caesar_salad", "chocolate_cake", "donuts", "dumplings",
    "french_fries", "fried_rice", "grilled_salmon", "hamburger",
    "hot_dog", "ice_cream", "omelette", "pancakes",
    "pizza", "ramen", "sushi",
]

# ---------- Ngưỡng Portion ----------
THRESHOLDS = {
    "small": 0.28,   # weighted_score < 0.28 → Small
    "large": 0.55,   # weighted_score > 0.55 → Large
}

LABEL_MAP = {0: "Small", 1: "Medium", 2: "Large"}
COLOR_MAP  = {"Small": "#3498db", "Medium": "#f39c12", "Large": "#e74c3c"}

SEED               = 42
N_SAMPLE_PER_CLASS = 100   # batch run
N_LABEL_PER_CLASS  = 10    # gán nhãn thủ công — 10 ảnh/class = 150 ảnh tổng

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ============================================================
# LOAD SAM-b MODEL (chạy 1 lần, tái dùng cho toàn bộ pipeline)
# ultralytics tự download weights sam_b.pt (~375 MB)
# ============================================================
sam_model = SAM("sam_b.pt")
sam_model.to(DEVICE)
print("✓ SAM-b loaded on", DEVICE)

In [ ]:
# ============================================================
# TIỀN XỬ LÝ — dùng lại từ giữa kỳ (Bilateral + CLAHE + Resize)
# ============================================================
def preprocess_image(img_bgr: np.ndarray, size: int = 512) -> np.ndarray:
    """
    Bilateral filter → CLAHE trên kênh L → Resize giữ tỷ lệ + pad reflect.
    Output: ảnh BGR 512×512.
    """
    filtered = cv2.bilateralFilter(img_bgr, d=5, sigmaColor=50, sigmaSpace=50)
    lab = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    h, w = enhanced.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(enhanced, (nw, nh), interpolation=cv2.INTER_LINEAR)

    pad_h, pad_w = size - nh, size - nw
    padded = cv2.copyMakeBorder(
        resized,
        pad_h // 2, pad_h - pad_h // 2,
        pad_w // 2, pad_w - pad_w // 2,
        cv2.BORDER_REFLECT,
    )
    return padded

In [ ]:
# ============================================================
# SEGMENTATION — SAM-b (thay thế GrabCut)
# ============================================================
def sam_segment(img_bgr: np.ndarray,
                model: SAM,
                n_grid: int = 3) -> tuple:
    """
    Dùng SAM-b với point prompt để tách vùng thức ăn.

    Chiến lược prompt:
      - Lưới n_grid × n_grid điểm foreground ở vùng trung tâm ảnh
        (thức ăn trong Food-101 thường nằm giữa khung hình)
      - 4 điểm góc làm negative prompt (background)
      → SAM chọn mask phủ tốt nhất các điểm foreground
        và loại trừ các điểm góc

    Trả về:
      binary_mask : np.ndarray uint8 (0/255), cùng kích thước img_bgr
      masked_img  : img_bgr với background=0
    """
    h, w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # --- Tạo lưới điểm foreground ở vùng trung tâm (40%–60% mỗi chiều) ---
    margin = 0.2   # bỏ 20% viền mỗi phía khi lấy điểm FG
    xs = np.linspace(w * margin, w * (1 - margin), n_grid, dtype=int)
    ys = np.linspace(h * margin, h * (1 - margin), n_grid, dtype=int)
    fg_points = [[int(x), int(y)] for y in ys for x in xs]   # (n²,2)
    fg_labels = [1] * len(fg_points)

    # --- 4 điểm góc làm background (negative) ---
    corner_offset = int(min(h, w) * 0.04)
    bg_points = [
        [corner_offset,     corner_offset],
        [w - corner_offset, corner_offset],
        [corner_offset,     h - corner_offset],
        [w - corner_offset, h - corner_offset],
    ]
    bg_labels = [0] * 4

    all_points = fg_points + bg_points
    all_labels = fg_labels + bg_labels

    # --- Chạy SAM ---
    try:
        results = model(
            img_rgb,
            points=[all_points],
            labels=[all_labels],
            verbose=False,
        )
        masks_tensor = results[0].masks  # Masks object
        if masks_tensor is not None and len(masks_tensor) > 0:
            masks_np = masks_tensor.data.cpu().numpy()  # (N, H, W) bool
            # Chọn mask có diện tích lớn nhất (foreground chính)
            areas = [m.sum() for m in masks_np]
            best  = masks_np[int(np.argmax(areas))]     # (H, W) bool
            binary = (best * 255).astype(np.uint8)
        else:
            # Fallback: toàn ảnh là foreground
            binary = np.full((h, w), 255, dtype=np.uint8)
    except Exception as e:
        print(f"  [SAM warning] {e} — dùng fallback mask")
        binary = np.full((h, w), 255, dtype=np.uint8)

    # --- Resize về kích thước gốc nếu SAM trả về shape khác ---
    if binary.shape != (h, w):
        binary = cv2.resize(binary, (w, h),
                            interpolation=cv2.INTER_NEAREST)

    # --- Morphology cleanup nhỏ để lấp lỗ và loại noise ---
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,
                              cv2.getStructuringElement(
                                  cv2.MORPH_ELLIPSE, (3, 3)))

    masked = cv2.bitwise_and(img_bgr, img_bgr, mask=binary)
    return binary, masked


def visualize_sam_prompt(img_bgr: np.ndarray,
                         binary: np.ndarray,
                         n_grid: int = 3) -> None:
    """
    Hiển thị ảnh gốc kèm các điểm prompt đã dùng + mask SAM.
    Dùng để debug khi cần kiểm tra prompt có đúng không.
    """
    h, w = img_bgr.shape[:2]
    margin = 0.2
    xs = np.linspace(w * margin, w * (1 - margin), n_grid, dtype=int)
    ys = np.linspace(h * margin, h * (1 - margin), n_grid, dtype=int)
    fg_points = [[int(x), int(y)] for y in ys for x in xs]
    corner_offset = int(min(h, w) * 0.04)
    bg_points = [
        [corner_offset,     corner_offset],
        [w - corner_offset, corner_offset],
        [corner_offset,     h - corner_offset],
        [w - corner_offset, h - corner_offset],
    ]

    overlay = img_bgr.copy()
    overlay[binary == 0] = (overlay[binary == 0] * 0.3).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    # Ảnh gốc + điểm prompt
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    for px, py in fg_points:
        axes[0].plot(px, py, 'g+', markersize=10, markeredgewidth=2)
    for px, py in bg_points:
        axes[0].plot(px, py, 'rx', markersize=10, markeredgewidth=2)
    axes[0].set_title("Prompt points\n(+ FG xanh | x BG đỏ)")
    axes[0].axis("off")
    # Mask overlay
    axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"SAM mask\n(fg={binary.sum()//255} px)")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Kiểm tra SAM trên 1 ảnh mẫu trước khi chạy toàn bộ ---
sample_cls  = "pizza"
sample_path = sorted((IMAGE_DIR / sample_cls).glob("*.jpg"))[0]
sample_bgr  = cv2.imread(str(sample_path))
sample_pre  = preprocess_image(sample_bgr)

mask_demo, _ = sam_segment(sample_pre, sam_model)
visualize_sam_prompt(sample_pre, mask_demo)
print(f"Area ratio demo: {mask_demo.sum() // 255 / sample_pre.shape[0]**2:.2%}")

In [ ]:
# ============================================================
# TRÍCH XUẤT RATIO FEATURES — giữ nguyên từ pipeline gốc
# ============================================================
def extract_ratio_features(img_bgr: np.ndarray,
                            mask: np.ndarray,
                            food_class: str = "") -> dict:
    """
    5 ratio từ mask SAM:
      R1 area_ratio   : fg_pixels / total_pixels
      R2 bbox_fill    : fg_pixels / bounding_box_area
      R3 compactness  : 4π·area / perimeter²
      R4 saturation   : mean HSV-S trong foreground
      R5 edge_density : cạnh Canny trong foreground / fg_pixels
    """
    h, w = img_bgr.shape[:2]
    total_pixels = h * w
    fg_pixels    = int(mask.sum() // 255)

    features = {"food_class": food_class,
                "total_pixels": total_pixels,
                "fg_pixels": fg_pixels}

    # R1
    r1 = fg_pixels / total_pixels if total_pixels > 0 else 0.0
    features["R1_area_ratio"] = round(float(r1), 4)

    # R2 + R3
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest   = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(largest)
        bbox_area = bw * bh
        r2 = fg_pixels / bbox_area if bbox_area > 0 else 0.0
        features["R2_bbox_fill"] = round(float(min(r2, 1.0)), 4)
        features["_bbox"]        = (x, y, bw, bh)

        perimeter    = cv2.arcLength(largest, closed=True)
        contour_area = cv2.contourArea(largest)
        r3 = (4 * np.pi * contour_area) / (perimeter ** 2) if perimeter > 0 else 0.0
        features["R3_compactness"] = round(float(min(r3, 1.0)), 4)
    else:
        features["R2_bbox_fill"]   = 0.0
        features["R3_compactness"] = 0.0
        features["_bbox"]          = (0, 0, w, h)

    # R4
    hsv         = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    fg_bool     = mask > 0
    r4 = float(hsv[:,:,1][fg_bool].mean()) / 255.0 if fg_bool.any() else 0.0
    features["R4_saturation"] = round(r4, 4)

    # R5
    gray  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    if fg_pixels > 0:
        edges_fg = cv2.bitwise_and(edges, edges, mask=mask)
        r5 = float(edges_fg.sum() // 255) / fg_pixels
    else:
        r5 = 0.0
    features["R5_edge_density"] = round(float(min(r5, 1.0)), 4)

    return features

In [ ]:
# ============================================================
# TÍNH PORTION SCORE & GÁN NHÃN S/M/L
# ============================================================
WEIGHTS = {
    "R1_area_ratio" : 0.50,
    "R2_bbox_fill"  : 0.25,
    "R3_compactness": 0.05,
    "R4_saturation" : 0.10,
    "R5_edge_density": 0.10,
}

CLASS_OFFSET = {
    "ramen"        :  0.08,
    "pizza"        :  0.06,
    "fried_rice"   :  0.05,
    "caesar_salad" : -0.05,
    "sushi"        : -0.03,
}


def compute_weighted_score(features: dict) -> float:
    return round(float(sum(
        WEIGHTS[k] * features.get(k, 0.0) for k in WEIGHTS
    )), 4)


def classify_portion(score: float, food_class: str = "") -> str:
    offset  = CLASS_OFFSET.get(food_class, 0.0)
    t_small = THRESHOLDS["small"] + offset
    t_large = THRESHOLDS["large"] + offset
    if score < t_small:  return "Small"
    if score > t_large:  return "Large"
    return "Medium"

In [ ]:
# ============================================================
# FULL PIPELINE CHO 1 ẢNH
# ============================================================
def run_portion_pipeline(img_bgr: np.ndarray,
                         food_class: str = "") -> dict:
    """
    Input : ảnh BGR (bất kỳ kích thước)
    Output: dict chứa toàn bộ kết quả
      - R1..R5        : 5 ratio features
      - weighted_score: điểm tổng hợp [0,1]
      - portion_label : 'Small' / 'Medium' / 'Large'
      - _mask         : binary mask uint8 (internal)
      - _preprocessed : ảnh đã preprocess (internal)
    """
    preprocessed          = preprocess_image(img_bgr, size=512)
    mask, masked          = sam_segment(preprocessed, sam_model)
    features              = extract_ratio_features(preprocessed, mask, food_class)
    score                 = compute_weighted_score(features)
    portion               = classify_portion(score, food_class)

    return {
        **features,
        "weighted_score" : score,
        "portion_label"  : portion,
        "_mask"          : mask,
        "_masked_img"    : masked,
        "_preprocessed"  : preprocessed,
    }

In [ ]:
# ============================================================
# BƯỚC 1 — BATCH RUN: xem phân bố prediction tổng thể
# ============================================================
def collect_image_paths(classes, n_per_class, seed=42):
    random.seed(seed)
    records = []
    for cls in classes:
        imgs    = sorted((IMAGE_DIR / cls).glob("*.jpg"))
        sampled = random.sample(imgs, min(n_per_class, len(imgs)))
        records += [{"path": str(p), "food_class": cls} for p in sampled]
    return records


def batch_run(classes, n_per_class=N_SAMPLE_PER_CLASS):
    records = collect_image_paths(classes, n_per_class, seed=SEED)
    results = []
    for rec in tqdm(records, desc="Batch SAM pipeline"):
        img_bgr = cv2.imread(rec["path"])
        if img_bgr is None:
            continue
        result = run_portion_pipeline(img_bgr, rec["food_class"])
        results.append({k: v for k, v in result.items()
                        if not k.startswith("_")})

    df = pd.DataFrame(results)
    df.to_csv(OUTPUT_DIR / "portion_predictions.csv", index=False)
    print(f"\n✓ Đã xử lý {len(df)} ảnh")
    print("\nPhân bố portion labels:")
    print(df["portion_label"].value_counts())
    print("\nPhân bố theo class:")
    print(df.groupby("food_class")["portion_label"]
            .value_counts().unstack(fill_value=0))
    return df


df_batch = batch_run(CLASSES, n_per_class=N_SAMPLE_PER_CLASS)

In [ ]:
# ============================================================
# BƯỚC 2 — GÁN NHÃN THỦ CÔNG
# 10 ảnh/class × 15 class = 150 ảnh tổng (~25-30 phút)
#
# Tiêu chí gán nhãn nhất quán:
#   Small  : thức ăn < 1/3 khung hình, hoặc 1 phần nhỏ
#   Medium : thức ăn chiếm 1/3 – 2/3 khung hình
#   Large  : thức ăn > 2/3 khung hình, hoặc cả đĩa/tô đầy
# ============================================================
def interactive_label(records, save_path):
    results = []
    for i, rec in enumerate(records):
        img_bgr = cv2.imread(rec["path"])
        if img_bgr is None:
            continue

        result = run_portion_pipeline(img_bgr, rec["food_class"])
        pred   = result["portion_label"]
        score  = result["weighted_score"]

        # Hiển thị 3 panel
        fig, axes = plt.subplots(1, 3, figsize=(13, 4))

        axes[0].imshow(cv2.cvtColor(result["_preprocessed"],
                                    cv2.COLOR_BGR2RGB))
        axes[0].set_title("Original")
        axes[0].axis("off")

        overlay = result["_preprocessed"].copy()
        overlay[result["_mask"] == 0] = (
            overlay[result["_mask"] == 0] * 0.3).astype(np.uint8)
        axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[1].set_title(
            f"SAM Mask\nArea ratio: {result['R1_area_ratio']:.2%}")
        axes[1].axis("off")

        ratio_names  = ["R1 Area", "R2 BBox", "R3 Compact",
                        "R4 Sat", "R5 Edges"]
        ratio_values = [result["R1_area_ratio"], result["R2_bbox_fill"],
                        result["R3_compactness"], result["R4_saturation"],
                        result["R5_edge_density"]]
        axes[2].barh(ratio_names, ratio_values,
                     color=COLOR_MAP[pred], alpha=0.8)
        axes[2].set_xlim(0, 1)
        axes[2].set_title(f"System: {pred}\nscore={score:.3f}")
        axes[2].axvline(0.5, color="gray", linestyle="--", alpha=0.5)

        fig.suptitle(
            f"[{i+1}/{len(records)}] {rec['food_class']}",
            fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()

        while True:
            user_input = input(
                f"  System: [{pred}] | Nhãn (S/M/L hoặc Enter đồng ý): "
            ).strip().upper()
            if user_input == "":
                label = pred; break
            elif user_input in ("S", "M", "L"):
                label = {"S":"Small","M":"Medium","L":"Large"}[user_input]
                break
            else:
                print("  → Chỉ nhập S, M, L hoặc Enter")

        results.append({
            **{k: v for k, v in result.items() if not k.startswith("_")},
            "manual_label": label,
            "path": rec["path"],
        })
        plt.close("all")

    df = pd.DataFrame(results)
    df.to_csv(save_path, index=False)
    print(f"\n✓ Đã lưu {len(df)} nhãn → {save_path}")
    return df


label_csv = OUTPUT_DIR / "manual_labels.csv"
if not label_csv.exists():
    records_to_label = collect_image_paths(
        CLASSES, N_LABEL_PER_CLASS, seed=SEED)
    df_labeled = interactive_label(records_to_label, label_csv)
else:
    print(f"✓ Đã có file nhãn: {label_csv}")
    df_labeled = pd.read_csv(label_csv)

In [ ]:
# ============================================================
# BƯỚC 3 — ĐÁNH GIÁ ĐỊNH LƯỢNG
# ============================================================
def evaluate_portion(df_labeled):
    y_true = df_labeled["manual_label"].tolist()
    y_pred = df_labeled["portion_label"].tolist()
    labels = ["Small", "Medium", "Large"]

    acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)
    print(f"\n{'='*50}")
    print(f"  PORTION ESTIMATION — KẾT QUẢ ĐÁNH GIÁ")
    print(f"{'='*50}")
    print(f"  Segmentation  : SAM-b (Segment Anything Model Base)")
    print(f"  Tổng ảnh      : {len(y_true)}  ({N_LABEL_PER_CLASS}/class)")
    print(f"  Accuracy      : {acc:.1%}")
    print(f"\n{classification_report(y_true, y_pred, labels=labels, zero_division=0)}")

    # --- Confusion matrix + per-class accuracy ---
    cm  = confusion_matrix(y_true, y_pred, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=axes[0], colorbar=False, cmap="Blues")
    axes[0].set_title("Confusion Matrix")

    class_acc = {cls: (df_labeled[df_labeled["food_class"] == cls]
                       .eval("manual_label == portion_label")
                       .mean())
                 for cls in CLASSES
                 if cls in df_labeled["food_class"].values}
    sorted_cls    = sorted(class_acc, key=class_acc.get)
    sorted_colors = ["#e74c3c" if class_acc[c] < 0.5 else
                     "#f39c12" if class_acc[c] < 0.7 else
                     "#27ae60" for c in sorted_cls]
    axes[1].barh(sorted_cls, [class_acc[c] for c in sorted_cls],
                 color=sorted_colors)
    axes[1].axvline(0.5,  color="red",  linestyle="--", alpha=0.7, label="50%")
    axes[1].axvline(acc,  color="navy", linestyle="-",  alpha=0.7,
                    label=f"Overall {acc:.1%}")
    axes[1].set_xlim(0, 1)
    axes[1].set_xlabel("Accuracy")
    axes[1].set_title("Accuracy theo class")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "evaluation.png", dpi=150, bbox_inches="tight")
    plt.show()

    # --- Lỗi phổ biến nhất ---
    errors = df_labeled[df_labeled["manual_label"] != df_labeled["portion_label"]]
    print("\n  Các trường hợp nhầm lẫn phổ biến:")
    print(errors.groupby(["manual_label","portion_label"])
                .size().reset_index(name="count")
                .sort_values("count", ascending=False)
                .to_string(index=False))

    # --- Histogram score distribution ---
    fig, ax = plt.subplots(figsize=(9, 4))
    for lbl in ["Small", "Medium", "Large"]:
        sub = df_labeled[df_labeled["manual_label"] == lbl]["weighted_score"]
        ax.hist(sub, bins=15, alpha=0.6, label=lbl, color=COLOR_MAP[lbl])
    ax.axvline(THRESHOLDS["small"], color="gray",  linestyle="--",
               label=f"T_small={THRESHOLDS['small']}")
    ax.axvline(THRESHOLDS["large"], color="black", linestyle="--",
               label=f"T_large={THRESHOLDS['large']}")
    ax.set_xlabel("Weighted Score")
    ax.set_ylabel("Số ảnh")
    ax.set_title("Phân bố Weighted Score theo nhãn thật")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "score_distribution.png",
                dpi=150, bbox_inches="tight")
    plt.show()


evaluate_portion(df_labeled)

In [ ]:
# ============================================================
# BƯỚC 4 — VISUALISATION
# ============================================================
def visualize_samples(df, n_per_portion=3):
    """Hiển thị n ảnh mỗi nhãn: original | SAM mask | ratio bars."""
    samples = pd.concat([
        df[df["portion_label"] == p].sample(
            min(n_per_portion, len(df[df["portion_label"] == p])),
            random_state=SEED)
        for p in ["Small", "Medium", "Large"]
    ]).reset_index(drop=True)

    n   = len(samples)
    fig, axes = plt.subplots(n, 3, figsize=(13, 4 * n))
    if n == 1: axes = [axes]

    for i, row in samples.iterrows():
        img_bgr = cv2.imread(row["path"])
        result  = run_portion_pipeline(img_bgr, row["food_class"])

        axes[i][0].imshow(cv2.cvtColor(result["_preprocessed"],
                                        cv2.COLOR_BGR2RGB))
        axes[i][0].set_title(row["food_class"]); axes[i][0].axis("off")

        overlay = result["_preprocessed"].copy()
        overlay[result["_mask"] == 0] = (
            overlay[result["_mask"] == 0] * 0.25).astype(np.uint8)
        axes[i][1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axes[i][1].set_title(
            f"SAM Mask\nArea: {result['R1_area_ratio']:.2%}")
        axes[i][1].axis("off")

        ratio_names  = ["R1 Area","R2 BBox","R3 Compact","R4 Sat","R5 Edges"]
        ratio_values = [result["R1_area_ratio"], result["R2_bbox_fill"],
                        result["R3_compactness"], result["R4_saturation"],
                        result["R5_edge_density"]]
        color = COLOR_MAP[result["portion_label"]]
        axes[i][2].barh(ratio_names, ratio_values, color=color, alpha=0.8)
        axes[i][2].set_xlim(0, 1)
        axes[i][2].set_title(
            f"Portion: {result['portion_label']}  "
            f"(score={result['weighted_score']:.3f})")
        for t in THRESHOLDS.values():
            axes[i][2].axvline(t, color="gray", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "portion_samples.png",
                dpi=150, bbox_inches="tight")
    plt.show()


def plot_ratio_analysis(df):
    """Boxplot 5 ratio theo nhãn thật — giúp chỉnh WEIGHTS."""
    ratio_cols   = ["R1_area_ratio","R2_bbox_fill","R3_compactness",
                    "R4_saturation","R5_edge_density"]
    ratio_labels = ["R1 Area","R2 BBox","R3 Compact","R4 Sat","R5 Edges"]

    fig, axes = plt.subplots(1, 5, figsize=(18, 5))
    for ax, col, lbl in zip(axes, ratio_cols, ratio_labels):
        data = [df[df["manual_label"] == p][col].dropna().values
                for p in ["Small","Medium","Large"]]
        bp   = ax.boxplot(data, labels=["S","M","L"], patch_artist=True)
        for patch, c in zip(bp["boxes"],
                            [COLOR_MAP["Small"],COLOR_MAP["Medium"],
                             COLOR_MAP["Large"]]):
            patch.set_facecolor(c); patch.set_alpha(0.7)
        ax.set_title(lbl); ax.set_ylim(0, 1)

    fig.suptitle("Phân bố từng Ratio theo nhãn khẩu phần",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ratio_boxplots.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("→ Ratio nào hộp S/M/L tách xa nhau nhất = đóng góp nhiều nhất.")
    print("  Tăng WEIGHTS của ratio đó để cải thiện accuracy.")


visualize_samples(df_labeled, n_per_portion=3)
plot_ratio_analysis(df_labeled)
print(f"\n✓ Hoàn thành! Kết quả lưu tại: {OUTPUT_DIR}")